<a href="https://colab.research.google.com/github/banikinkar/Assignments_Of_AI/blob/main/LangChain_Comparison_App_2026_07_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 LangChain Parity — Chunking Strategies

**Two tabs only:** Tokenization and a **LangChain Comparison** that checks each custom chunking
strategy against LangChain's equivalent on the same short / medium / large inputs.

For strategies where LangChain has a true equivalent (Token, Paragraph, Character) the tab runs
**both** and reports an **EXACT MATCH** badge computed live. For custom strategies LangChain has no
equivalent for (word-budget, sentence-count, token-packing), the tab says so plainly and shows the
custom output. Nothing is hardcoded — every result is computed when you click.

_Built 2026-07-16. Run top to bottom in Colab; the last cell prints a public share link._

## 1. Install dependencies

In [ ]:
!pip install -q gradio tiktoken langchain-text-splitters -U
print('\u2713 dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 27.1 MB/s eta 0:00:00
✓ dependencies installed


## 2. Imports & constants

In [ ]:
import re
import logging
from typing import List, Dict, Any, Tuple, Callable, Optional
import tiktoken
from langchain_text_splitters import (
    CharacterTextSplitter,
    TokenTextSplitter,
    RecursiveCharacterTextSplitter,
)

# LangChain's CharacterTextSplitter warns when a split exceeds chunk_size; harmless here, silence it.
logging.getLogger('langchain_text_splitters.base').setLevel(logging.ERROR)
logging.getLogger('langchain.text_splitter').setLevel(logging.ERROR)

MINILM_HARD_LIMIT = 256
MINILM_SAFE_LIMIT = 240
MAX_TEXT_SIZE = 50000  # 50KB
print('\u2713 imports loaded')

✓ imports loaded


## 3. Token counter (tiktoken cl100k_base)

In [ ]:
class MiniLMTokenCounter:
    """Token counter with detailed breakdown support."""

    def __init__(self):
        self.encoding = tiktoken.get_encoding("cl100k_base")

    def count(self, text: str) -> int:
        return len(self.encoding.encode(text))

    def encode(self, text: str) -> List[int]:
        return self.encoding.encode(text)

    def decode(self, tokens: List[int]) -> str:
        return self.encoding.decode(tokens)

    def get_token_details(self, text: str) -> List[Dict[str, Any]]:
        tokens = self.encode(text)
        details = []
        byte_offset = 0
        for i, token_id in enumerate(tokens):
            decoded = self.decode([token_id])
            byte_length = len(decoded.encode('utf-8'))
            details.append({
                "token_id": token_id,
                "token_text": decoded,
                "byte_offset": byte_offset,
                "byte_length": byte_length,
                "index": i,
            })
            byte_offset += byte_length
        return details

counter = MiniLMTokenCounter()
print(f"\u2713 Token counter initialized (safe_limit={MINILM_SAFE_LIMIT})")

✓ Token counter initialized (safe_limit=240)


## 4. Custom chunking strategies (verbatim — the tested implementations)

In [ ]:
class UnifiedChunker:
    """All 8 strategies in one class (identical to the Comprehensive app)."""

    def __init__(self, token_counter: MiniLMTokenCounter):
        self.counter = token_counter

    @staticmethod
    def sentences(text: str) -> List[str]:
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

    def fixed_size_chunking(self, text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
        words = text.split()
        chunks = []
        overlap = min(overlap, chunk_size // 4)
        step = max(1, chunk_size - overlap)
        for i in range(0, len(words), step):
            chunk_words = words[i : i + chunk_size]
            if chunk_words:
                chunks.append(" ".join(chunk_words))
        return chunks

    def sentence_based_chunking(self, text: str, sentences_per_chunk: int = 10) -> List[str]:
        sentences = self.sentences(text)
        chunks = []
        for i in range(0, len(sentences), sentences_per_chunk):
            chunk_sents = sentences[i : i + sentences_per_chunk]
            if chunk_sents:
                chunks.append(" ".join(chunk_sents))
        return chunks

    def sliding_window_chunking(self, text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
        words = text.split()
        chunks = []
        overlap = min(overlap, chunk_size // 5)
        step = max(1, chunk_size - overlap)
        for i in range(0, len(words), step):
            chunk_words = words[i : i + chunk_size]
            if chunk_words:
                chunks.append(" ".join(chunk_words))
        return chunks

    def paragraph_level_chunking(self, text: str) -> List[str]:
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        if not paragraphs:
            paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
        return paragraphs or [text]

    def character_text_splitter(self, text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
        chunks = []
        overlap = min(overlap, chunk_size // 10)
        step = max(1, chunk_size - overlap)
        for i in range(0, len(text), step):
            chunk = text[i : i + chunk_size]
            if chunk:
                chunks.append(chunk)
        return chunks

    def token_text_splitter(self, text: str, max_tokens: int = 100, overlap_tokens: int = 20) -> List[str]:
        tokens = self.counter.encode(text)
        overlap_tokens = min(overlap_tokens, max_tokens // 5)
        step = max(1, max_tokens - overlap_tokens)
        chunks = []
        for i in range(0, len(tokens), step):
            chunk_tokens = tokens[i : i + max_tokens]
            if chunk_tokens:
                chunks.append(self.counter.decode(chunk_tokens))
        return chunks

    def _token_windows(self, text: str, target_tokens: int, overlap_tokens: int = 20) -> List[str]:
        tokens = self.counter.encode(text)
        step = max(1, target_tokens - overlap_tokens)
        chunks = []
        for start in range(0, len(tokens), step):
            chunk_tokens = tokens[start : start + target_tokens]
            if chunk_tokens:
                chunks.append(self.counter.decode(chunk_tokens))
        return chunks

    def _pack_units(self, units: List[str], target_tokens: int) -> List[str]:
        if not units:
            return []
        chunks: List[str] = []
        current: List[str] = []
        current_tokens = 0
        for unit in units:
            unit_tokens = self.counter.count(unit)
            if unit_tokens > target_tokens:
                if current:
                    chunks.append(" ".join(current))
                    current = []
                    current_tokens = 0
                chunks.extend(self._token_windows(unit, target_tokens))
                continue
            separator_tokens = 1 if current else 0
            if current and current_tokens + separator_tokens + unit_tokens > target_tokens:
                chunks.append(" ".join(current))
                current = [unit]
                current_tokens = unit_tokens
            else:
                current.append(unit)
                current_tokens = self.counter.count(" ".join(current))
        if current:
            chunks.append(" ".join(current))
        return [c for c in chunks if c.strip()]

    def recursive_text_splitter(self, text: str, target_tokens: int = MINILM_SAFE_LIMIT) -> List[str]:
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        units: List[str] = []
        for para in paragraphs or [text]:
            units.extend(self.sentences(para) or [para])
        return self._pack_units(units, target_tokens)

    def hierarchical_chunking(self, text: str, target_tokens: int = MINILM_SAFE_LIMIT) -> List[str]:
        section_pattern = r"\n(?=[A-Z][A-Z\s]+\n|\d+\.\s|\n#)"
        units: List[str] = []
        for section in re.split(section_pattern, text):
            section = section.strip()
            if not section:
                continue
            paragraphs = [p.strip() for p in section.split("\n\n") if p.strip()]
            for para in paragraphs or [section]:
                units.extend(self.sentences(para) or [para])
        return self._pack_units(units, target_tokens)

chunker = UnifiedChunker(counter)
print("\u2713 Unified chunker with 8 strategies loaded")

✓ Unified chunker with 8 strategies loaded


### 4b. Verbatim source of each strategy (shown in the comparison tab)

In [ ]:
SOURCE = {
    '__init__': 'def __init__(self, token_counter: MiniLMTokenCounter):\n        self.counter = token_counter',
    'sentences': 'def sentences(text: str) -> List[str]:\n        return [s.strip() for s in re.split(r"(?<=[.!?])\\s+", text.strip()) if s.strip()]',
    'fixed_size_chunking': 'def fixed_size_chunking(self, text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:\n        words = text.split()\n        chunks = []\n        overlap = min(overlap, chunk_size // 4)\n        step = max(1, chunk_size - overlap)\n        for i in range(0, len(words), step):\n            chunk_words = words[i : i + chunk_size]\n            if chunk_words:\n                chunks.append(" ".join(chunk_words))\n        return chunks',
    'sentence_based_chunking': 'def sentence_based_chunking(self, text: str, sentences_per_chunk: int = 10) -> List[str]:\n        sentences = self.sentences(text)\n        chunks = []\n        for i in range(0, len(sentences), sentences_per_chunk):\n            chunk_sents = sentences[i : i + sentences_per_chunk]\n            if chunk_sents:\n                chunks.append(" ".join(chunk_sents))\n        return chunks',
    'sliding_window_chunking': 'def sliding_window_chunking(self, text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:\n        words = text.split()\n        chunks = []\n        overlap = min(overlap, chunk_size // 5)\n        step = max(1, chunk_size - overlap)\n        for i in range(0, len(words), step):\n            chunk_words = words[i : i + chunk_size]\n            if chunk_words:\n                chunks.append(" ".join(chunk_words))\n        return chunks',
    'paragraph_level_chunking': 'def paragraph_level_chunking(self, text: str) -> List[str]:\n        paragraphs = [p.strip() for p in text.split("\\n\\n") if p.strip()]\n        if not paragraphs:\n            paragraphs = [p.strip() for p in text.split("\\n") if p.strip()]\n        return paragraphs or [text]',
    'character_text_splitter': 'def character_text_splitter(self, text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:\n        chunks = []\n        overlap = min(overlap, chunk_size // 10)\n        step = max(1, chunk_size - overlap)\n        for i in range(0, len(text), step):\n            chunk = text[i : i + chunk_size]\n            if chunk:\n                chunks.append(chunk)\n        return chunks',
    'token_text_splitter': 'def token_text_splitter(self, text: str, max_tokens: int = 100, overlap_tokens: int = 20) -> List[str]:\n        tokens = self.counter.encode(text)\n        overlap_tokens = min(overlap_tokens, max_tokens // 5)\n        step = max(1, max_tokens - overlap_tokens)\n        chunks = []\n        for i in range(0, len(tokens), step):\n            chunk_tokens = tokens[i : i + max_tokens]\n            if chunk_tokens:\n                chunks.append(self.counter.decode(chunk_tokens))\n        return chunks',
    '_token_windows': 'def _token_windows(self, text: str, target_tokens: int, overlap_tokens: int = 20) -> List[str]:\n        tokens = self.counter.encode(text)\n        step = max(1, target_tokens - overlap_tokens)\n        chunks = []\n        for start in range(0, len(tokens), step):\n            chunk_tokens = tokens[start : start + target_tokens]\n            if chunk_tokens:\n                chunks.append(self.counter.decode(chunk_tokens))\n        return chunks',
    '_pack_units': 'def _pack_units(self, units: List[str], target_tokens: int) -> List[str]:\n        if not units:\n            return []\n        chunks: List[str] = []\n        current: List[str] = []\n        current_tokens = 0\n        for unit in units:\n            unit_tokens = self.counter.count(unit)\n            if unit_tokens > target_tokens:\n                if current:\n                    chunks.append(" ".join(current))\n                    current = []\n                    current_tokens = 0\n                chunks.extend(self._token_windows(unit, target_tokens))\n                continue\n            separator_tokens = 1 if current else 0\n            if current and current_tokens + separator_tokens + unit_tokens > target_tokens:\n                chunks.append(" ".join(current))\n                current = [unit]\n                current_tokens = unit_tokens\n            else:\n                current.append(unit)\n                current_tokens = self.counter.count(" ".join(current))\n        if current:\n            chunks.append(" ".join(current))\n        return [c for c in chunks if c.strip()]',
    'recursive_text_splitter': 'def recursive_text_splitter(self, text: str, target_tokens: int = MINILM_SAFE_LIMIT) -> List[str]:\n        paragraphs = [p.strip() for p in text.split("\\n\\n") if p.strip()]\n        units: List[str] = []\n        for para in paragraphs or [text]:\n            units.extend(self.sentences(para) or [para])\n        return self._pack_units(units, target_tokens)',
    'hierarchical_chunking': 'def hierarchical_chunking(self, text: str, target_tokens: int = MINILM_SAFE_LIMIT) -> List[str]:\n        section_pattern = r"\\n(?=[A-Z][A-Z\\s]+\\n|\\d+\\.\\s|\\n#)"\n        units: List[str] = []\n        for section in re.split(section_pattern, text):\n            section = section.strip()\n            if not section:\n                continue\n            paragraphs = [p.strip() for p in section.split("\\n\\n") if p.strip()]\n            for para in paragraphs or [section]:\n                units.extend(self.sentences(para) or [para])\n        return self._pack_units(units, target_tokens)',
}
print(f"\u2713 SOURCE dict built ({len(SOURCE)} methods)")


✓ SOURCE dict built (12 methods)


## 5. Sample texts (short / medium / large)

In [ ]:
SAMPLE_TEXTS = {
    "Short Example": (
        "Machine learning transforms software development. "
        "AI automates previously manual tasks. "
        "The future is intelligent and adaptive."
    ),
    "Medium Example (Healthcare)": (
        "Artificial Intelligence in Healthcare\n\n"
        "INTRODUCTION\n\n"
        "AI revolutionizes healthcare by enabling faster diagnoses and personalized treatment. "
        "Machine learning analyzes medical data at scale.\n\n"
        "APPLICATIONS\n\n"
        "Medical imaging detects anomalies in X-rays, MRIs, and CT scans. "
        "Predictive analytics forecasts disease progression. "
        "NLP extracts insights from clinical notes."
    ),
    "Long Example (Oversized \u2014 316 tokens)": (
        "Artificial Intelligence is revolutionizing the healthcare industry by enabling faster, more accurate diagnoses "
        "and personalized treatment plans. Machine learning algorithms can analyze complex medical data, identify patterns, "
        "and provide insights that were previously impossible to extract at scale. "
        "Deep learning systems trained on millions of medical images can now detect certain cancers with accuracy rates "
        "that rival or exceed human radiologists. These AI-powered diagnostic tools are deployed in hospitals worldwide, "
        "reducing diagnosis time from weeks to minutes and enabling earlier intervention. "
        "Beyond diagnosis, AI transforms treatment planning through predictive analytics, forecasting disease progression. "
        "Natural language processing extracts insights from unstructured clinical notes, enabling population health strategies. "
        "Advanced machine learning models are now capable of analyzing genetic data to predict disease susceptibility decades before symptoms appear. "
        "This predictive capability enables preventive medicine strategies that can significantly reduce healthcare costs and improve patient outcomes. "
        "Furthermore, AI-driven drug discovery has accelerated the development of new pharmaceuticals, reducing time to market from years to months. "
        "Computer vision systems can now identify subtle patterns in medical imaging that human radiologists might miss, leading to earlier diagnoses. "
        "Natural language models process patient records to identify at-risk populations and recommend interventions before complications develop. "
        "Machine learning algorithms optimize hospital operations, from staffing decisions to equipment maintenance schedules, improving efficiency. "
        "Telemedicine platforms powered by AI provide diagnosis and treatment recommendations in remote areas with limited access to specialists. "
        "Blockchain technology integrated with AI systems ensures data security and patient privacy while enabling collaborative research across institutions. "
        "Regulatory frameworks are evolving to accommodate these innovations while ensuring patient safety and ethical implementation. "
        "Healthcare organizations are investing heavily in AI infrastructure to remain competitive and improve patient outcomes."
    ),
}

def validate_input(text: str) -> Tuple[bool, str]:
    if not text or not text.strip():
        return False, "\u274c Text cannot be empty. Please enter some text."
    if len(text) > MAX_TEXT_SIZE:
        return False, f"\u274c Text too large ({len(text)} chars). Max: {MAX_TEXT_SIZE} chars (~50KB)."
    if counter.count(text) < 2:
        return False, "\u274c Text must be at least 2 tokens. Please add more content."
    return True, "\u2713 Input valid"

print("Sample texts ready:")
for name, text in SAMPLE_TEXTS.items():
    print(f"  \u2022 {name}: {counter.count(text)} tokens")

Sample texts ready:
  • Short Example: 20 tokens
  • Medium Example (Healthcare): 62 tokens
  • Long Example (Oversized — 316 tokens): 316 tokens


## 6. Custom \u2194 LangChain equivalence map

Each custom strategy is mapped to either a **LangChain equivalent** (both are run and compared) or
flagged **custom-only** with the reason LangChain cannot reproduce it. When an equivalent exists, the
custom strategy's *effective* parameters (after its own clamping) are mirrored into the LangChain
splitter so the comparison isolates the algorithm, not the config.

In [ ]:
# Default params, mirrored into LangChain so the comparison is apples-to-apples.
FIXED_WORDS = 100
CHAR_SIZE = 500
TOKEN_MAX = 100
PARENT_TOKENS = MINILM_SAFE_LIMIT   # large parent chunk (matches custom hierarchical's 240-token budget)
CHILD_TOKENS = 60                   # small child chunk indexed for precise matching

def lc_parent_child(text: str):
    """Faithful, offline reproduction of LangChain's parent-child (small-to-big) structure.

    Mirrors ParentDocumentRetriever internals: a parent_splitter makes large parents, then a
    child_splitter cuts each parent into small children that reference their parent. Returns a
    list of (parent_text, [child_texts]) — the two-level structure, no vectorstore/API needed.
    """
    seps = ["\n\n", "\n", ". ", " ", ""]
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=PARENT_TOKENS, chunk_overlap=0, length_function=counter.count, separators=seps)
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHILD_TOKENS, chunk_overlap=0, length_function=counter.count, separators=seps)
    parents = parent_splitter.split_text(text)
    return [(p, child_splitter.split_text(p)) for p in parents]

def lc_paragraph(text: str, overlap: int) -> List[str]:
    # custom paragraph_level = split on blank line, strip, drop empties
    return CharacterTextSplitter(
        separator="\n\n", chunk_size=1, chunk_overlap=0,
        strip_whitespace=True, keep_separator=False,
    ).split_text(text)

def lc_character(text: str, overlap: int) -> List[str]:
    # custom character_text_splitter: raw char window, overlap clamped to size//10
    eff_overlap = min(overlap, CHAR_SIZE // 10)
    return CharacterTextSplitter(
        separator="", chunk_size=CHAR_SIZE, chunk_overlap=eff_overlap,
        strip_whitespace=False, keep_separator=False,
    ).split_text(text)

def lc_token(text: str, overlap: int) -> List[str]:
    # custom token_text_splitter: cl100k_base windows, overlap clamped to max//5
    eff_overlap = min(overlap, TOKEN_MAX // 5)
    return TokenTextSplitter(
        encoding_name="cl100k_base", chunk_size=TOKEN_MAX, chunk_overlap=eff_overlap,
    ).split_text(text)

def lc_recursive(text: str, overlap: int) -> List[str]:
    # custom recursive_text_splitter packs paragraph->sentence units to a TOKEN budget
    # (MINILM_SAFE_LIMIT). Mirror that: RecursiveCharacterTextSplitter measuring length in
    # TOKENS (not characters) with the same budget, recursing paragraph -> line -> sentence -> word.
    return RecursiveCharacterTextSplitter(
        chunk_size=MINILM_SAFE_LIMIT, chunk_overlap=0,
        length_function=counter.count,
        separators=["\n\n", "\n", ". ", " ", ""],
    ).split_text(text)

# strategy_name -> dict(custom=fn, langchain=fn_or_None, note=str, lc_name=str)
STRATEGIES: Dict[str, Dict[str, Any]] = {
    "Token-Based": {
        "custom": lambda t, ov: chunker.token_text_splitter(t, overlap_tokens=ov),
        "langchain": lc_token,
        "method": "token_text_splitter",
        "lc_name": "TokenTextSplitter(cl100k_base, size=100)",
        "note": "Both slice cl100k_base token windows with the same step \u2192 identical output.",
        "code_locus": "token_text_splitter(): step = max_tokens - overlap_tokens; window tokens[i:i+max_tokens]",
        "expected": "Expected IDENTICAL. Any difference here would be UNEXPECTED \u2014 investigate.",
    },
    "Paragraph-Level": {
        "custom": lambda t, ov: chunker.paragraph_level_chunking(t),
        "langchain": lc_paragraph,
        "method": "paragraph_level_chunking",
        "lc_name": "CharacterTextSplitter(separator='\\n\\n')",
        "note": "Both split on blank lines and strip \u2192 identical output.",
        "code_locus": "paragraph_level_chunking(): text.split('\\n\\n') then strip and drop empties",
        "expected": "Expected IDENTICAL. Any difference here would be UNEXPECTED \u2014 investigate.",
    },
    "Character-Level": {
        "custom": lambda t, ov: chunker.character_text_splitter(t, overlap=ov),
        "langchain": lc_character,
        "method": "character_text_splitter",
        "lc_name": "CharacterTextSplitter(separator='', size=500)",
        "note": "Both slide a raw character window; last-chunk/overlap tail handling can differ slightly.",
        "code_locus": "character_text_splitter(): fixed step = chunk_size - overlap over text[i:i+chunk_size]",
        "expected": "May differ ONLY at the final chunk: LangChain's _merge_splits stops early once the remaining characters can't fill a full window, so its last chunk can be shorter. NORMAL and non-fatal \u2014 no characters are dropped.",
    },
    "Fixed-Size": {
        "custom": lambda t, ov: chunker.fixed_size_chunking(t, overlap=ov),
        "langchain": None,
        "method": "fixed_size_chunking",
        "lc_name": "\u2014",
        "note": "Custom budgets by WORD count. LangChain has no word-count splitter (it budgets by characters or tokens).",
    },
    "Sliding Window": {
        "custom": lambda t, ov: chunker.sliding_window_chunking(t, overlap=ov),
        "langchain": None,
        "method": "sliding_window_chunking",
        "lc_name": "\u2014",
        "note": "Word-budget sliding window. No LangChain equivalent (LangChain budgets by characters/tokens).",
    },
    "Sentence-Based": {
        "custom": lambda t, ov: chunker.sentence_based_chunking(t),
        "langchain": None,
        "method": "sentence_based_chunking",
        "lc_name": "\u2014",
        "note": "Groups a fixed COUNT of sentences. LangChain's NLTK/Spacy splitters re-merge by character size, not sentence count.",
    },
    "Recursive": {
        "custom": lambda t, ov: chunker.recursive_text_splitter(t),
        "langchain": lc_recursive,
        "method": "recursive_text_splitter",
        "lc_name": "RecursiveCharacterTextSplitter(token length, budget=240)",
        "note": "Both recurse paragraph\u2192line\u2192sentence\u2192word and pack to a 240-token budget. Custom joins units with a single space (collapsing blank lines) and keeps sentence-final periods attached; LangChain preserves original separators (e.g. blank lines) and splits on '. '.",
        "code_locus": "recursive_text_splitter() \u2192 _pack_units(): units joined with ' '.join(); sentence split via regex (?<=[.!?])\\s+",
        "expected": "Differences are whitespace/separator placement only (blank lines vs spaces, period adjacency). Same text content, same 240-token budget \u2192 NORMAL, non-fatal.",
    },
    "Hierarchical": {
        "custom": lambda t, ov: chunker.hierarchical_chunking(t),
        "langchain": None,
        "kind": "parent_child",
        "parent_child": lc_parent_child,
        "method": "hierarchical_chunking",
        "lc_name": "ParentDocumentRetriever (parent 240-tok / child 60-tok, small-to-big)",
        "note": "NOT the same kind of thing. Custom hierarchical is a FLAT splitter (1 level): it walks document structure (HEADINGS / numbered / '#' \u2192 paragraph \u2192 sentence) then token-packs to 240 tokens. LangChain parent-child is a 2-LEVEL retrieval structure: large parent chunks stored for context + small child chunks indexed for matching, each child linked to its parent. Different outputs \u2014 shown side-by-side, not diffed as a match.",
        "code_locus": "hierarchical_chunking() \u2192 _pack_units(): returns a flat List[str]; there is no child level or parent linkage to compare against.",
    },
}
_flat = sum(1 for s in STRATEGIES.values() if s.get("langchain"))
_pc = sum(1 for s in STRATEGIES.values() if s.get("kind") == "parent_child")
_custom_only = len(STRATEGIES) - _flat - _pc
print(f"\u2713 {len(STRATEGIES)} strategies mapped: {_flat} with a flat LangChain equivalent, "
      f"{_pc} parent-child (structural), {_custom_only} custom-only")

✓ 8 strategies mapped: 4 with a flat LangChain equivalent, 1 parent-child (structural), 3 custom-only


## 7. Gradio app \u2014 Tokenization + LangChain Comparison

In [ ]:
import gradio as gr
import html as _html
import difflib
from collections import Counter as _Multiset

STRAT_CHOICES = list(STRATEGIES.keys())

# Helper methods some strategies rely on — shown alongside the main method source.
_HELPER_METHODS = {
    "recursive_text_splitter": ["_pack_units", "_token_windows", "sentences"],
    "hierarchical_chunking": ["_pack_units", "_token_windows", "sentences"],
    "sentence_based_chunking": ["sentences"],
}

def custom_source(strategy: str) -> str:
    """Return the verbatim custom source (from the SOURCE dict) for a strategy's method(s)."""
    spec = STRATEGIES[strategy]
    method = spec.get("method")
    if not method or method not in SOURCE:
        return "(source unavailable)"
    parts = [SOURCE[method].rstrip()]
    for helper in _HELPER_METHODS.get(method, []):
        if helper in SOURCE:
            parts.append(SOURCE[helper].rstrip())
    return "\n\n".join(parts)

def _code_block(strategy: str) -> str:
    src = _html.escape(custom_source(strategy))
    return (
        "<details style='margin:10px 0'><summary style='cursor:pointer;font-weight:bold;color:#0d47a1'>"
        f"\U0001F4DC Show custom code for &lt;{_html.escape(strategy)}&gt;</summary>"
        f"<pre style='background:#0d1117;color:#e6edf3;padding:12px;border-radius:6px;overflow-x:auto;"
        f"font-size:12px;line-height:1.4'>{src}</pre></details>"
    )

def analyze_difference(strategy: str, custom_chunks, lc_chunks) -> str:
    """Locate the first divergence, attribute it to the code, and grade severity — all live."""
    spec = STRATEGIES[strategy]
    if custom_chunks == lc_chunks:
        return ("<div style='padding:12px;border-radius:6px;background:#e6ffe6;color:#1b5e20'>"
                "\u2705 <strong>No differences.</strong> Every chunk is byte-for-byte identical.</div>")

    n = min(len(custom_chunks), len(lc_chunks))
    first = next((i for i in range(n) if custom_chunks[i] != lc_chunks[i]), n)
    last_index = max(len(custom_chunks), len(lc_chunks)) - 1
    only_final = all(
        (i >= n) or (custom_chunks[i] == lc_chunks[i]) or (i == last_index)
        for i in range(max(len(custom_chunks), len(lc_chunks)))
    )

    # Content-coverage checks → detect real data loss vs cosmetic differences.
    # (1) whitespace/separator-insensitive: same text once all whitespace is removed?
    _strip = lambda chunks: "".join("".join(chunks).split())
    whitespace_only = _strip(custom_chunks) == _strip(lc_chunks)
    # (2) word SET (ignores overlap duplication) → catches dropped words.
    cw = set(" ".join(custom_chunks).split())
    lw = set(" ".join(lc_chunks).split())
    lost_from_lc = cw - lw       # present in custom, missing from langchain
    lost_from_custom = lw - cw   # present in langchain, missing from custom
    content_preserved = whitespace_only or (not lost_from_lc and not lost_from_custom)

    # Severity grade
    if whitespace_only:
        grade = ("\U0001F7E2 NORMAL \u2014 whitespace/separator only", "#e6ffe6", "#1b5e20",
                 "The two outputs are the SAME text once whitespace and separators are ignored. "
                 "Only blank-line vs space and separator placement differ \u2014 no content changes. Non-fatal.")
    elif content_preserved and only_final:
        grade = ("\U0001F7E2 NORMAL \u2014 non-fatal", "#e6ffe6", "#1b5e20",
                 "Difference is confined to the final chunk boundary and NO content is lost. "
                 "This is expected overlap/tail handling, safe to ignore for retrieval.")
    elif content_preserved:
        grade = ("\U0001F7E1 STRUCTURAL \u2014 lossless", "#fff3e0", "#e65100",
                 "Chunks are grouped differently but the SET of words is identical on both sides \u2014 "
                 "no content is dropped. Retrieval quality is preserved; only chunk boundaries move.")
    else:
        grade = ("\u26d4 POTENTIALLY FATAL \u2014 content differs", "#ffe6e6", "#b71c1c",
                 "The two outputs do NOT cover the same content \u2014 one side is missing text. "
                 "This could drop information from retrieval and must be investigated.")

    label, bg, col, meaning = grade

    # Char-level diff of the first differing pair
    a = custom_chunks[first] if first < len(custom_chunks) else "(no chunk)"
    b = lc_chunks[first] if first < len(lc_chunks) else "(no chunk)"
    sm = difflib.SequenceMatcher(None, a, b)
    hi = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            hi.append(_html.escape(a[i1:i2]))
        else:
            if a[i1:i2]:
                hi.append(f"<span style='background:#ffd6d6;color:#b71c1c'>{_html.escape(a[i1:i2])}</span>")
            if b[j1:j2]:
                hi.append(f"<span style='background:#d6ffd6;color:#1b5e20'>{_html.escape(b[j1:j2])}</span>")
    diff_html = "".join(hi)

    def _lost(s):
        items = list(s)[:8]
        more = "\u2026" if len(s) > 8 else ""
        return _html.escape(", ".join(repr(x) for x in items)) + more if items else "(none)"

    return (
        f"<div style='padding:12px;border-radius:6px;background:{bg};color:{col};margin:8px 0'>"
        f"<strong>Severity: {label}</strong><br>{meaning}</div>"
        f"<table style='border-collapse:collapse;margin:8px 0;color:#000'>"
        f"<tr><td style='padding:4px 10px'><strong>First divergence</strong></td>"
        f"<td style='padding:4px 10px'>chunk #{first+1}"
        f"{' (final chunk only)' if only_final else ''}</td></tr>"
        f"<tr><td style='padding:4px 10px'><strong>Chunk counts</strong></td>"
        f"<td style='padding:4px 10px'>custom={len(custom_chunks)}, langchain={len(lc_chunks)}</td></tr>"
        f"<tr><td style='padding:4px 10px'><strong>Content preserved?</strong></td>"
        f"<td style='padding:4px 10px'>"
        f"{'YES \u2014 same text ignoring whitespace' if whitespace_only else ('YES \u2014 same word set' if content_preserved else 'NO \u2014 words differ')}"
        f"</td></tr>"
        f"<tr><td style='padding:4px 10px'><strong>Words only in custom</strong></td>"
        f"<td style='padding:4px 10px'>{_lost(lost_from_lc)}</td></tr>"
        f"<tr><td style='padding:4px 10px'><strong>Words only in langchain</strong></td>"
        f"<td style='padding:4px 10px'>{_lost(lost_from_custom)}</td></tr>"
        f"<tr><td style='padding:4px 10px'><strong>Where in code</strong></td>"
        f"<td style='padding:4px 10px'><code>{_html.escape(spec.get('code_locus','\u2014'))}</code></td></tr>"
        f"<tr><td style='padding:4px 10px'><strong>Expected behaviour</strong></td>"
        f"<td style='padding:4px 10px'>{_html.escape(spec.get('expected','\u2014'))}</td></tr>"
        f"</table>"
        f"<p style='color:#000;margin:6px 0'><strong>First differing chunk (#{first+1}) \u2014 "
        f"<span style='background:#ffd6d6'>red = custom only</span>, "
        f"<span style='background:#d6ffd6'>green = langchain only</span>:</strong></p>"
        f"<pre style='background:#fafafa;color:#000;padding:10px;border:1px solid #ddd;border-radius:6px;"
        f"white-space:pre-wrap;font-size:12px'>{diff_html}</pre>"
    )

def get_current_text(sample_preset: str, custom_text: str) -> str:
    return custom_text.strip() if custom_text.strip() else SAMPLE_TEXTS.get(sample_preset, "")

def format_sidebar_initial() -> str:
    result = "\U0001F4C4 ALL SAMPLE TEXTS (pick one or paste your own):\n\n" + "=" * 60 + "\n\n"
    for i, (name, text) in enumerate(SAMPLE_TEXTS.items(), 1):
        result += f"[{i}] {name} ({counter.count(text)} tokens)\n" + "\u2500" * 60 + "\n" + text + "\n\n" + "=" * 60 + "\n\n"
    return result

def format_sidebar_selected(text: str) -> str:
    if not text:
        return format_sidebar_initial()
    return f"\U0001F4C4 CURRENT INPUT ({counter.count(text)} tokens):\n\n{text}"

# ----------------------------- Tab: Tokenization -----------------------------
def tab_tokenization(sample_preset, custom_text, strategy, overlap):
    text = get_current_text(sample_preset, custom_text)
    valid, msg = validate_input(text)
    if not valid:
        return msg, ""
    tokens = counter.encode(text)
    details = counter.get_token_details(text)
    try:
        chunks = STRATEGIES[strategy]["custom"](text, overlap)
    except Exception as e:
        return f"\u274c Error: {e}", ""

    token_to_chunk = {}
    pos = 0
    for ci, chunk in enumerate(chunks):
        for _ in counter.encode(chunk):
            if pos < len(tokens):
                token_to_chunk[pos] = ci
                pos += 1

    parts = []
    for ci, chunk in enumerate(chunks):
        ct = counter.count(chunk)
        over = ct > MINILM_HARD_LIMIT
        color = "#ff4444" if over else "#44ff44"
        bg = "#ffe6e6" if over else "#e6ffe6"
        status = "\U0001F534" if over else "\U0001F7E2"
        safe = _html.escape(chunk[:200])
        parts.append(f'<div style="margin:10px 0;padding:10px;border-left:4px solid {color};color:#000;background:{bg}">')
        parts.append(f'<strong>{status} Chunk {ci+1}</strong>: {ct} tokens<br>')
        parts.append(f'<code style="font-size:12px;color:#000">{safe}{"..." if len(chunk)>200 else ""}</code></div>')
    html_viz = "".join(parts)

    max_ct = max([counter.count(c) for c in chunks]) if chunks else 0
    over_n = sum(1 for c in chunks if counter.count(c) > MINILM_HARD_LIMIT)
    breakdown = (
        f"\U0001F4CA TOKEN BREAKDOWN (ALL {len(tokens)} TOKENS)\n" + "=" * 70 + "\n\n"
        f"Total Tokens: {len(tokens)}\nOverlap: {overlap}\nChunks: {len(chunks)}\n"
        f"Max Chunk: {max_ct} tokens\nOversized (>256): {over_n}\n\n" + "=" * 70 + "\n"
        f"Index | Token ID | Token Text       | Chunk\n" + "-" * 70 + "\n"
    )
    for d in details:
        cn = token_to_chunk.get(d["index"], -1)
        label = f"Chunk {cn+1}" if cn >= 0 else "N/A"
        tt = d["token_text"].replace("\n", "\\n").replace("\r", "\\r")[:16].ljust(16)
        breakdown += f"{d['index']:<5} | {d['token_id']:<8} | {tt} | {label}\n"
    breakdown += "\n" + "=" * 70 + f"\n\u2713 All {len(tokens)} tokens displayed (no truncation)\n"
    return breakdown, html_viz

# ------------------------- Tab: LangChain Comparison -------------------------
def _side_by_side(custom_chunks, lc_chunks):
    n = max(len(custom_chunks), len(lc_chunks))
    rows = []
    for i in range(n):
        c = custom_chunks[i] if i < len(custom_chunks) else None
        l = lc_chunks[i] if i < len(lc_chunks) else None
        same = (c is not None and c == l)
        mark = "\u2705" if same else "\u26a0\ufe0f"
        cbg = "#e6ffe6" if same else "#fff3e0"
        def cell(x):
            return "<em style='color:#999'>(none)</em>" if x is None else _html.escape(x[:220]) + ("\u2026" if len(x) > 220 else "")
        rows.append(
            f"<tr style='background:{cbg}'>"
            f"<td style='padding:6px;border:1px solid #ccc;text-align:center;color:#000'>{mark} {i+1}</td>"
            f"<td style='padding:6px;border:1px solid #ccc;font-size:12px;color:#000'><code>{cell(c)}</code></td>"
            f"<td style='padding:6px;border:1px solid #ccc;font-size:12px;color:#000'><code>{cell(l)}</code></td></tr>"
        )
    header = ("<table style='border-collapse:collapse;width:100%'>"
              "<tr style='background:#222;color:#fff'>"
              "<th style='padding:6px;border:1px solid #ccc'>#</th>"
              "<th style='padding:6px;border:1px solid #ccc'>Custom</th>"
              "<th style='padding:6px;border:1px solid #ccc'>LangChain</th></tr>")
    return header + "".join(rows) + "</table>"

def _render_parent_child(strategy, custom_chunks, pc):
    """Structural side-by-side: custom flat chunks vs LangChain parent->child tree."""
    spec = STRATEGIES[strategy]
    n_parents = len(pc)
    n_children = sum(len(ch) for _, ch in pc)

    banner = (
        "<div style='padding:14px;border-radius:8px;background:#e3f2fd;color:#0d47a1;font-size:15px'>"
        "\U0001F333 <strong>Different by design \u2014 NOT a match.</strong> "
        "Custom hierarchical is a <strong>flat</strong> splitter (1 level). LangChain parent-child is a "
        "<strong>2-level</strong> retrieval structure (parent + child + linkage). Shown side-by-side so the "
        "difference is explicit.</div>"
    )
    verdict = (
        "<table style='border-collapse:collapse;margin:10px 0;color:#000'>"
        "<tr style='background:#222;color:#fff'><th style='padding:6px;border:1px solid #ccc'></th>"
        "<th style='padding:6px;border:1px solid #ccc'>Custom hierarchical</th>"
        "<th style='padding:6px;border:1px solid #ccc'>LangChain parent-child</th></tr>"
        f"<tr><td style='padding:6px;border:1px solid #ccc'><strong>Levels</strong></td>"
        f"<td style='padding:6px;border:1px solid #ccc'>1 (flat chunks)</td>"
        f"<td style='padding:6px;border:1px solid #ccc'>2 (parent \u2192 child)</td></tr>"
        f"<tr><td style='padding:6px;border:1px solid #ccc'><strong>Units</strong></td>"
        f"<td style='padding:6px;border:1px solid #ccc'>{len(custom_chunks)} chunks</td>"
        f"<td style='padding:6px;border:1px solid #ccc'>{n_parents} parents / {n_children} children</td></tr>"
        f"<tr><td style='padding:6px;border:1px solid #ccc'><strong>Parent\u2013child linkage</strong></td>"
        f"<td style='padding:6px;border:1px solid #ccc'>none</td>"
        f"<td style='padding:6px;border:1px solid #ccc'>yes (each child \u2192 parent_id)</td></tr>"
        f"<tr><td style='padding:6px;border:1px solid #ccc'><strong>Purpose</strong></td>"
        f"<td style='padding:6px;border:1px solid #ccc'>one set of embeddable chunks</td>"
        f"<td style='padding:6px;border:1px solid #ccc'>small children match, big parents give context</td></tr>"
        "</table>"
    )
    left = "<h4 style='color:#000'>\U0001F4C4 Custom hierarchical \u2014 flat chunks (1 level)</h4>"
    for i, c in enumerate(custom_chunks):
        left += (f"<div style='margin:6px 0;padding:8px;border-left:4px solid #1976d2;background:#f5f5f5;color:#000'>"
                 f"<strong>Chunk {i+1}</strong> ({counter.count(c)} tok)<br>"
                 f"<code style='font-size:12px'>{_html.escape(c[:180])}{'\u2026' if len(c)>180 else ''}</code></div>")
    right = "<h4 style='color:#000'>\U0001F333 LangChain parent-child \u2014 2-level tree</h4>"
    for pi, (parent, children) in enumerate(pc):
        right += (f"<div style='margin:8px 0;padding:8px;border:2px solid #2e7d32;border-radius:6px;background:#e8f5e9;color:#000'>"
                  f"<strong>PARENT P{pi+1}</strong> ({counter.count(parent)} tok, stored for context)<br>"
                  f"<code style='font-size:12px'>{_html.escape(parent[:160])}{'\u2026' if len(parent)>160 else ''}</code>")
        for ci, child in enumerate(children):
            right += (f"<div style='margin:4px 0 4px 18px;padding:6px;border-left:3px solid #66bb6a;background:#fff;color:#000'>"
                      f"\u21B3 <strong>child P{pi+1}.{ci+1}</strong> ({counter.count(child)} tok) "
                      f"<span style='color:#888'>parent_id=P{pi+1}</span><br>"
                      f"<code style='font-size:11px'>{_html.escape(child[:120])}{'\u2026' if len(child)>120 else ''}</code></div>")
        right += "</div>"
    two_col = (f"<div style='display:flex;gap:16px;flex-wrap:wrap'>"
               f"<div style='flex:1;min-width:280px'>{left}</div>"
               f"<div style='flex:1;min-width:280px'>{right}</div></div>")

    detail = (banner + f"<p style='color:#000;margin-top:8px'>{_html.escape(spec['note'])}</p>"
              + verdict + _code_block(strategy) + two_col)
    summary = (
        f"Strategy: {strategy}\n"
        f"LangChain equivalent: {spec['lc_name']}\n"
        f"Custom = FLAT (1 level): {len(custom_chunks)} chunks\n"
        f"LangChain parent-child = 2 levels: {n_parents} parents, {n_children} children (linked)\n"
        f"Verdict: different by design \u2014 not comparable as a flat match.\n\n{spec['note']}"
    )
    return summary, detail

def compare_one(sample_preset, custom_text, strategy, overlap):
    text = get_current_text(sample_preset, custom_text)
    valid, msg = validate_input(text)
    if not valid:
        return msg, ""
    spec = STRATEGIES[strategy]
    custom_chunks = spec["custom"](text, overlap)
    code_html = _code_block(strategy)

    if spec.get("kind") == "parent_child":
        return _render_parent_child(strategy, custom_chunks, spec["parent_child"](text))

    if spec["langchain"] is None:
        banner = (
            f"<div style='padding:14px;border-radius:8px;background:#e3f2fd;color:#0d47a1;font-size:15px'>"
            f"\U0001F9E9 <strong>{strategy} \u2014 custom-only strategy.</strong> "
            f"No standard LangChain equivalent to compare against.<br>"
            f"<span style='color:#333'>{_html.escape(spec['note'])}</span></div>"
        )
        parts = [banner, code_html,
                 f"<p style='color:#000'><strong>Custom output \u2014 {len(custom_chunks)} chunks:</strong></p>"]
        for i, c in enumerate(custom_chunks):
            parts.append(
                f"<div style='margin:6px 0;padding:8px;border-left:4px solid #1976d2;background:#f5f5f5;color:#000'>"
                f"<strong>Chunk {i+1}</strong> ({counter.count(c)} tokens)<br>"
                f"<code style='font-size:12px'>{_html.escape(c[:220])}{'\u2026' if len(c)>220 else ''}</code></div>"
            )
        return "\U0001F9E9 Custom-only strategy (no LangChain equivalent to compare).", "".join(parts)

    lc_chunks = spec["langchain"](text, overlap)
    exact = custom_chunks == lc_chunks
    per_chunk_ok = sum(1 for i in range(min(len(custom_chunks), len(lc_chunks))) if custom_chunks[i] == lc_chunks[i])

    if exact:
        badge = (f"<div style='padding:14px;border-radius:8px;background:#e6ffe6;color:#1b5e20;font-size:16px'>"
                 f"\u2705 <strong>EXACT MATCH</strong> \u2014 custom and LangChain produced identical output "
                 f"({len(custom_chunks)} chunks).</div>")
    else:
        badge = (f"<div style='padding:14px;border-radius:8px;background:#fff3e0;color:#e65100;font-size:16px'>"
                 f"\u26a0\ufe0f <strong>DIFFERS</strong> \u2014 custom={len(custom_chunks)} chunks, "
                 f"LangChain={len(lc_chunks)} chunks; {per_chunk_ok} identical before first divergence.</div>")

    summary = (
        f"Strategy: {strategy}\n"
        f"LangChain equivalent: {spec['lc_name']}\n"
        f"Effective overlap mirrored into LangChain: yes\n"
        f"Custom chunks: {len(custom_chunks)} | LangChain chunks: {len(lc_chunks)}\n"
        f"Exact match: {exact}\n\n{spec['note']}"
    )
    detail = (
        badge
        + f"<p style='color:#000;margin-top:10px'>{_html.escape(spec['note'])}</p>"
        + code_html
        + "<h4 style='color:#000;margin:12px 0 4px'>\U0001F50D Difference analysis</h4>"
        + analyze_difference(strategy, custom_chunks, lc_chunks)
        + "<h4 style='color:#000;margin:12px 0 4px'>Side-by-side chunks</h4>"
        + _side_by_side(custom_chunks, lc_chunks)
    )
    return summary, detail

def compare_all(sample_preset, custom_text, overlap):
    text = get_current_text(sample_preset, custom_text)
    valid, msg = validate_input(text)
    if not valid:
        return msg
    rows = []
    for name, spec in STRATEGIES.items():
        custom_chunks = spec["custom"](text, overlap)
        if spec.get("kind") == "parent_child":
            pc = spec["parent_child"](text)
            n_parents = len(pc); n_children = sum(len(ch) for _, ch in pc)
            status = "\U0001F333 2-level (structural)"
            detail = f"custom flat={len(custom_chunks)} vs parents={n_parents}/children={n_children}"
            bg = "#e3f2fd"; col = "#0d47a1"
        elif spec["langchain"] is None:
            status = "\U0001F9E9 custom-only"
            detail = "no LangChain equivalent"
            bg = "#e3f2fd"; col = "#0d47a1"
        else:
            lc_chunks = spec["langchain"](text, overlap)
            if custom_chunks == lc_chunks:
                status = "\u2705 EXACT MATCH"
                detail = f"{len(custom_chunks)} chunks identical"
                bg = "#e6ffe6"; col = "#1b5e20"
            else:
                status = "\u26a0\ufe0f differs"
                detail = f"custom={len(custom_chunks)}, lc={len(lc_chunks)}"
                bg = "#fff3e0"; col = "#e65100"
        rows.append(
            f"<tr style='background:{bg}'>"
            f"<td style='padding:8px;border:1px solid #ccc;color:#000'><strong>{name}</strong></td>"
            f"<td style='padding:8px;border:1px solid #ccc;color:{col};font-weight:bold'>{status}</td>"
            f"<td style='padding:8px;border:1px solid #ccc;color:#000;font-size:12px'>{spec['lc_name']}</td>"
            f"<td style='padding:8px;border:1px solid #ccc;color:#000;font-size:12px'>{detail}</td></tr>"
        )
    return ("<table style='border-collapse:collapse;width:100%'>"
            "<tr style='background:#222;color:#fff'>"
            "<th style='padding:8px;border:1px solid #ccc'>Custom strategy</th>"
            "<th style='padding:8px;border:1px solid #ccc'>Result</th>"
            "<th style='padding:8px;border:1px solid #ccc'>LangChain equivalent</th>"
            "<th style='padding:8px;border:1px solid #ccc'>Detail</th></tr>" + "".join(rows) + "</table>")

def create_app():
    with gr.Blocks(title="LangChain Parity \u2014 Chunking") as demo:
        gr.Markdown("# \U0001F52C LangChain Parity \u2014 Chunking Strategies")
        gr.Markdown("Custom strategies vs LangChain equivalents on the same short / medium / large inputs. "
                    "All match results are computed live.")
        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                sidebar_text = gr.Textbox(label="\U0001F4C4 Sample Texts", lines=25,
                                          interactive=False, value=format_sidebar_initial())
            with gr.Column(scale=3):
                with gr.Row():
                    with gr.Column(scale=2):
                        sample_preset = gr.Dropdown(choices=list(SAMPLE_TEXTS.keys()),
                                                    value="Long Example (Oversized \u2014 316 tokens)",
                                                    label="\U0001F4CB Sample Preset")
                        custom_text = gr.Textbox(lines=8, label="Or paste your own text",
                                                 placeholder="Enter text to chunk...")
                    with gr.Column(scale=1):
                        strategy = gr.Dropdown(choices=STRAT_CHOICES, value="Token-Based",
                                               label="\u2699\ufe0f Strategy")
                        overlap_slider = gr.Slider(minimum=0, maximum=50, step=5, value=20,
                                                   label="\U0001F517 Overlap (mirrored into LangChain)")
                        gr.Markdown(
                            "<sub>\U0001F517 <strong>Overlap applies to:</strong> "
                            "<strong>Sliding Window</strong> (words \u2014 defines the window), "
                            "<strong>Fixed-Size</strong> (words), <strong>Character-Level</strong> (chars), "
                            "<strong>Token-Based</strong> (tokens).<br>"
                            "<strong>No effect on:</strong> Sentence-Based, Paragraph-Level, Recursive, "
                            "Hierarchical \u2014 these chunk by structure or a token budget, so the slider "
                            "is ignored.</sub>"
                        )

                def update_sidebar(sample, custom):
                    return format_sidebar_selected(custom.strip()) if custom.strip() \
                        else format_sidebar_selected(SAMPLE_TEXTS.get(sample, ""))
                sample_preset.change(update_sidebar, [sample_preset, custom_text], sidebar_text)
                custom_text.change(update_sidebar, [sample_preset, custom_text], sidebar_text)

                with gr.Tabs():
                    with gr.Tab("\U0001F524 Tokenization"):
                        out_tok = gr.Textbox(lines=20, label="All Tokens (no truncation)", interactive=False)
                        out_viz = gr.HTML(label="Visual")
                        gr.Button("Analyze").click(tab_tokenization,
                            [sample_preset, custom_text, strategy, overlap_slider], [out_tok, out_viz])

                    with gr.Tab("\u2696\ufe0f LangChain Comparison"):
                        gr.Markdown("### All strategies at a glance")
                        out_all = gr.HTML()
                        gr.Button("Compare ALL strategies").click(compare_all,
                            [sample_preset, custom_text, overlap_slider], out_all)
                        gr.Markdown("### Selected strategy \u2014 side-by-side")
                        out_cmp_sum = gr.Textbox(lines=8, label="Summary", interactive=False)
                        out_cmp = gr.HTML()
                        gr.Button("Compare selected strategy").click(compare_one,
                            [sample_preset, custom_text, strategy, overlap_slider], [out_cmp_sum, out_cmp])
    return demo

print("\u2713 App defined")

✓ App defined


## 8. Launch (Colab: use the public share link)

In [ ]:
app = create_app()
app.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5742e14cfaa64cd81.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
